## Exp1: CGM-only baseline (resolved subset)

In [ ]:
import sys
from pathlib import Path

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/dissertation')
else:
    PROJECT_ROOT = Path('../../..').resolve()

sys.path.insert(0, str(PROJECT_ROOT))

print(f"IN_COLAB={IN_COLAB}  REPO_ROOT={PROJECT_ROOT}")

Mounted at /content/drive
IN_COLAB=True  REPO_ROOT=/content/drive/MyDrive/dissertation


In [15]:
import json

import numpy as np

from src import dataloader as data
from src import metrics
from src import train
from src.models.classical_baseline import HORIZONS
from src.models.nn_baseline import GRUPredictor

MODEL_NAME = 'CGM-only (resolved)'
SEEDS = [7, 14, 21, 28, 35, 42, 49, 56, 63, 70]
ENCODER_NAME = None

EXPERIMENT_DIR = PROJECT_ROOT / 'experiments' / '3_multimodal' / 'multimodal'
RESULTS_DIR = EXPERIMENT_DIR / 'results' / 'cgm_only'
STABILITY_SUMMARY_PATH = RESULTS_DIR / 'cgm_only_stability_summary.json'

In [13]:
stats = data.load_cgm_stats()
train_loader = data.load_multimodal_tensors('train', encoder_name=ENCODER_NAME)
val_loader = data.load_multimodal_tensors('val', encoder_name=ENCODER_NAME, shuffle=False)
test_loader = data.load_multimodal_tensors('test', encoder_name=ENCODER_NAME, shuffle=False)

test_participant_ids = data.load_multimodal_participant_ids('test')
PARTICIPANT_IDS = sorted(set(test_participant_ids))

print('number of train batches: ', len(train_loader))
print('number of valuation batches: ', len(val_loader))
print('number of test batches:', len(test_loader))
print('\nnumber of test participants: ', len(PARTICIPANT_IDS))
print('corresponding participants: \n', PARTICIPANT_IDS)

sample_batch = next(iter(train_loader))
print('\nbatch shapes:', [tuple(x.shape) for x in sample_batch])

number of train batches:  323
number of valuation batches:  64
number of test batches: 66

number of test participants:  6
corresponding participants: 
 [np.str_('CGMacros-008'), np.str_('CGMacros-014'), np.str_('CGMacros-019'), np.str_('CGMacros-020'), np.str_('CGMacros-034'), np.str_('CGMacros-039')]

batch shapes: [(256, 24), (256,), (256,), (256, 5)]


In [14]:
def forward_fn(model, batch, device):
    cgm, tsm, has_meal, target = batch
    return model(cgm.to(device)), target.to(device)

In [ ]:
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
all_results = []  # one entry per seed, in memory

for seed in SEEDS:
    print(f"Test results — {MODEL_NAME} | seed={seed}\n")

    train.set_seed(seed)  # before constructing the model
    model = GRUPredictor(hidden_size=256, num_layers=3, dropout=0.2)

    model, history = train.train_model(
        model, train_loader, val_loader, stats['cgm_mean'], stats['cgm_std'],
        lr=0.0001, weight_decay=1e-05, forward_fn=forward_fn,
        checkpoint_dir=EXPERIMENT_DIR / 'checkpoints' / 'cgm_only' / f'cgm_only_seed{seed}',
        verbose=False, progress_bar=True, progress_desc=f'seed={seed}',
    )
    n_epochs = len(history['train_loss'])

    y_pred_val, y_true_val = train.predict(model, val_loader, forward_fn=forward_fn)
    y_pred_test, y_true_test = train.predict(model, test_loader, forward_fn=forward_fn)
    val_results = metrics.evaluate(y_pred_val, y_true_val, stats['cgm_mean'], stats['cgm_std'])
    test_results = metrics.evaluate(y_pred_test, y_true_test, stats['cgm_mean'], stats['cgm_std'])
    test_results_per_participant = metrics.evaluate_per_participant(
        y_pred_test, y_true_test, test_participant_ids, stats['cgm_mean'], stats['cgm_std'])

    seed_result = {
        'model': MODEL_NAME, 'seed': seed, 'n_epochs_trained': n_epochs,
        'val_results': val_results, 'test_results': test_results,
        'test_results_per_participant': test_results_per_participant,
    }
    all_results.append(seed_result)

    with open(RESULTS_DIR / f'cgm_only_seed{seed}_results.json', 'w') as f:
        json.dump(seed_result, f, indent=2)

    for h in HORIZONS:
        r = test_results[h]
        zone_a = r['ceg']['A']
        zone_ab = r['ceg']['A'] + r['ceg']['B']

        print(
            f"{h:>3} min | "
            f"RMSE {r['rmse']:.2f} | "
            f"MAE {r['mae']:.2f} | "
            f"gRMSE {r['grmse']:.2f} | "
            f"Zone A {zone_a:.2f}% | "
            f"A+B {zone_ab:.2f}%"
        )

    print(f"\n{'-' * 80}")

Test results — CGM-only (resolved) | seed=7

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


seed=7:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.17 | MAE 7.63 | gRMSE 12.27 | Zone A 96.95% | A+B 99.76%
 30 min | RMSE 18.47 | MAE 12.36 | gRMSE 21.34 | Zone A 90.33% | A+B 99.48%
 60 min | RMSE 27.38 | MAE 18.22 | gRMSE 33.40 | Zone A 81.39% | A+B 98.94%
 90 min | RMSE 32.82 | MAE 21.95 | gRMSE 40.92 | Zone A 75.31% | A+B 98.36%
120 min | RMSE 35.85 | MAE 24.20 | gRMSE 45.12 | Zone A 71.57% | A+B 98.06%

--------------------------------------------------------------------------------
Test results — CGM-only (resolved) | seed=14

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


seed=14:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.14 | MAE 7.53 | gRMSE 12.26 | Zone A 97.14% | A+B 99.80%
 30 min | RMSE 18.39 | MAE 12.11 | gRMSE 21.28 | Zone A 90.84% | A+B 99.55%
 60 min | RMSE 27.46 | MAE 18.01 | gRMSE 33.50 | Zone A 81.63% | A+B 98.92%
 90 min | RMSE 32.98 | MAE 21.83 | gRMSE 41.15 | Zone A 75.51% | A+B 98.23%
120 min | RMSE 36.03 | MAE 24.13 | gRMSE 45.42 | Zone A 71.54% | A+B 97.87%

--------------------------------------------------------------------------------
Test results — CGM-only (resolved) | seed=21

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


...

 15 min | RMSE 11.19 | MAE 7.56 | gRMSE 12.41 | Zone A 97.15% | A+B 99.79%
 30 min | RMSE 18.50 | MAE 12.21 | gRMSE 21.44 | Zone A 90.71% | A+B 99.53%
 60 min | RMSE 27.49 | MAE 18.09 | gRMSE 33.56 | Zone A 81.64% | A+B 98.94%
 90 min | RMSE 32.95 | MAE 21.97 | gRMSE 41.00 | Zone A 75.21% | A+B 98.31%
120 min | RMSE 35.92 | MAE 24.29 | gRMSE 45.06 | Zone A 71.37% | A+B 98.06%

--------------------------------------------------------------------------------
Test results — CGM-only (resolved) | seed=63

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


seed=63:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.29 | MAE 7.67 | gRMSE 12.33 | Zone A 96.97% | A+B 99.80%
 30 min | RMSE 18.62 | MAE 12.43 | gRMSE 21.48 | Zone A 90.44% | A+B 99.56%
 60 min | RMSE 27.68 | MAE 18.31 | gRMSE 33.85 | Zone A 81.35% | A+B 98.92%
 90 min | RMSE 33.15 | MAE 22.00 | gRMSE 41.49 | Zone A 75.25% | A+B 98.31%
120 min | RMSE 36.33 | MAE 24.25 | gRMSE 45.98 | Zone A 71.82% | A+B 97.90%

--------------------------------------------------------------------------------
Test results — CGM-only (resolved) | seed=70

device: cuda (NVIDIA A100-SXM4-40GB)  model: GRUPredictor  params: 1,021,957


seed=70:   0%|          | 0/50 [00:00<?, ?it/s]

 15 min | RMSE 11.21 | MAE 7.52 | gRMSE 12.38 | Zone A 97.18% | A+B 99.81%
 30 min | RMSE 18.49 | MAE 12.22 | gRMSE 21.39 | Zone A 90.53% | A+B 99.54%
 60 min | RMSE 27.45 | MAE 18.21 | gRMSE 33.43 | Zone A 81.28% | A+B 98.93%
 90 min | RMSE 32.87 | MAE 21.98 | gRMSE 40.88 | Zone A 75.07% | A+B 98.32%
120 min | RMSE 35.97 | MAE 24.33 | gRMSE 45.18 | Zone A 71.22% | A+B 98.01%

--------------------------------------------------------------------------------


In [ ]:
# Test Results
def _extract(participant_result, metric):
    if metric == 'zone_a':
        return participant_result['ceg']['A']
    if metric == 'zone_ab':
        return participant_result['ceg']['A'] + participant_result['ceg']['B']
    return participant_result[metric]

METRICS = ['rmse', 'mae', 'grmse', 'zone_a', 'zone_ab']

summary = {}

for h in HORIZONS:
    summary[h] = {}
    for m in METRICS:
        if m == 'zone_a':
            seed_values = [r['test_results'][h]['ceg']['A'] for r in all_results]
        elif m == 'zone_ab':
            seed_values = [r['test_results'][h]['ceg']['A'] + r['test_results'][h]['ceg']['B']
                            for r in all_results]
        else:
            seed_values = [r['test_results'][h][m] for r in all_results]

        mean = float(np.mean(seed_values))
        sd1 = float(np.std(seed_values, ddof=1))

        participant_means = []
        for pid in PARTICIPANT_IDS:
            values_across_seeds = [_extract(r['test_results_per_participant'][pid][h], m)
                                    for r in all_results]
            participant_means.append(np.mean(values_across_seeds))
        sd2 = float(np.std(participant_means, ddof=1))
        summary[h][m] = {'mean': mean, 'sd1_across_seeds': sd1, 'sd2_across_participants': sd2}

print(f'=== {MODEL_NAME} test set: overall mean +/- SD_seed (SD_participant) ===')
for h in HORIZONS:
    s = summary[h]
    print(
        f"{h:>3}-min:  "
        f"RMSE = {s['rmse']['mean']:.2f} +/- {s['rmse']['sd1_across_seeds']:.2f} "
        f"({s['rmse']['sd2_across_participants']:.2f}) mg/dL   "
        f"MAE = {s['mae']['mean']:.2f} +/- {s['mae']['sd1_across_seeds']:.2f} "
        f"({s['mae']['sd2_across_participants']:.2f})   "
        f"gRMSE = {s['grmse']['mean']:.2f} +/- {s['grmse']['sd1_across_seeds']:.2f} "
        f"({s['grmse']['sd2_across_participants']:.2f})   "
        f"Zone A = {s['zone_a']['mean']:.2f} +/- {s['zone_a']['sd1_across_seeds']:.2f} "
        f"({s['zone_a']['sd2_across_participants']:.2f})%   "
        f"Zone A+B = {s['zone_ab']['mean']:.2f} +/- {s['zone_ab']['sd1_across_seeds']:.2f} "
        f"({s['zone_ab']['sd2_across_participants']:.2f})%"
    )

with open(STABILITY_SUMMARY_PATH, 'w') as f:
    json.dump({
        'model': MODEL_NAME, 'seeds': SEEDS, 'n_seeds': len(SEEDS),
        'participant_ids': PARTICIPANT_IDS, 'per_horizon_summary': summary,
    }, f, indent=2)

print(f'\nsaved per-seed results to {RESULTS_DIR}')
print(f'saved aggregate summary to {STABILITY_SUMMARY_PATH}')

=== CGM-only (resolved) test set: overall mean +/- SD_seed (SD_participant) ===
 15-min:  RMSE = 11.21 +/- 0.09 (1.88) mg/dL   MAE = 7.61 +/- 0.12 (1.46)   gRMSE = 12.39 +/- 0.19 (2.97)   Zone A = 97.09 +/- 0.08 (2.04)%   Zone A+B = 99.79 +/- 0.02 (0.34)%
 30-min:  RMSE = 18.52 +/- 0.09 (3.87) mg/dL   MAE = 12.30 +/- 0.11 (2.66)   gRMSE = 21.47 +/- 0.20 (6.39)   Zone A = 90.56 +/- 0.15 (3.14)%   Zone A+B = 99.53 +/- 0.03 (0.60)%
 60-min:  RMSE = 27.54 +/- 0.19 (9.23) mg/dL   MAE = 18.18 +/- 0.10 (5.86)   gRMSE = 33.64 +/- 0.36 (14.41)   Zone A = 81.40 +/- 0.16 (5.16)%   Zone A+B = 98.92 +/- 0.04 (1.08)%
 90-min:  RMSE = 33.00 +/- 0.20 (13.15) mg/dL   MAE = 21.96 +/- 0.06 (8.21)   gRMSE = 41.16 +/- 0.36 (20.33)   Zone A = 75.15 +/- 0.18 (6.98)%   Zone A+B = 98.30 +/- 0.08 (1.74)%
120-min:  RMSE = 36.05 +/- 0.22 (15.65) mg/dL   MAE = 24.25 +/- 0.07 (10.06)   gRMSE = 45.39 +/- 0.38 (24.10)   Zone A = 71.41 +/- 0.23 (7.54)%   Zone A+B = 97.96 +/- 0.10 (2.60)%

saved per-seed results to /co